In [16]:
import os
os.environ["USE_TF"] = "0"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

In [17]:
from transformers import AutoModelForCausalLM , AutoTokenizer , TrainingArguments 
from peft import LoraConfig , get_peft_model , prepare_model_for_kbit_training 
from trl import SFTTrainer
from datasets import load_from_disk
import torch

In [18]:
MODEL_PATH   = r"C:\datasets\llm-data\models\phi3-base"
DATASET_PATH = r"C:\datasets\llm-data\pretrain\tokenized_dataset"
OUTPUT_PATH  = r"C:\datasets\llm-data\models\phi3-pretrained"

In [19]:
tokenizer = AutoTokenizer.from_pretrained(
    r"C:\datasets\llm-data\models\phi3-base",
    local_files_only=True,
    trust_remote_code=True
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print("Tokenizer loaded!")
print(tokenizer)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Tokenizer loaded!
LlamaTokenizerFast(name_or_path='C:\datasets\llm-data\models\phi3-base', vocab_size=32000, model_max_length=4096, is_fast=True, padding_side='left', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '<|endoftext|>', 'unk_token': '<unk>', 'pad_token': '<|endoftext|>'}, clean_up_tokenization_spaces=False),  added_tokens_decoder={
	0: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("</s>", rstrip=True, lstrip=False, single_word=False, normalized=False, special=False),
	32000: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	32001: AddedToken("<|assistant|>", rstrip=True, lstrip=False, single_word=False, normalized=False, special=True),
	32002: AddedToken("<|placeholder1|>", rstrip=True, lstrip=False, single_word=False, nor

In [20]:
print("Loading Model...")
model = AutoModelForCausalLM.from_pretrained(MODEL_PATH,torch_dtype=torch.float16,device_map="auto",local_files_only=True)

Loading Model...


In [21]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj","k_proj",
        "v_proj","o_proj"
    ],
    lora_dropout=0.05,
    bias=None,
    task_type= "CAUSAL_LM",
    inference_mode=False
)

In [22]:
print("Applying LoRA...")
model.enable_input_require_grads()
model = get_peft_model(model,lora_config)
model.print_trainable_parameters()

Applying LoRA...


NotImplementedError: Requested bias: None, is not implemented.

In [ ]:
print("Loading dataset...")
train_dataset = load_from_disk(f"{DATASET_PATH}/train")
val_dataset   = load_from_disk(f"{DATASET_PATH}/val")

In [ ]:
import os
os.environ["USE_TF"] = "0"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import transformers, peft, trl, accelerate
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("trl:", trl.__version__)
print("accelerate:", accelerate.__version__)

transformers: 4.41.0
peft: 0.11.0
trl: 0.9.2
accelerate: 0.30.0
